In [ ]:
from FIT_python.pipeline_sex.sex_config import run_otter_search_sex, run_other_species_search


In [ ]:
# Run quick searches for demonstration purposes
run_otter_search_sex(n_iter=1, cv=2)
run_other_species_search(n_iter=1, cv=2)


In [ ]:
import pandas as pd
from joblib import load
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from FIT_python.config import SPLITS_DIR, RESULTS_DATA_DIR

def evaluate(species, metric='balanced_test_acc'):
    base_dir = RESULTS_DATA_DIR / f'{species}_random_search_standard_metrics'
    model_path = base_dir / f'best_{metric}' / f'{species}.joblib'
    clf = load(model_path)
    df = pd.read_parquet(SPLITS_DIR / species / 'test.parquet')
    df = df.query("sex in ['f','m']").drop(columns=['Fold'], errors='ignore')
    feature_cols = [c for c in df.columns if c not in ['id','species','individual_id','trail','sex','Fold','date','location','dataorigin','substrate']]
    X = df[feature_cols]
    y = df['sex'].map({'f':0,'m':1}).values
    pred = clf.predict(X)
    return {'species': species, 'accuracy': accuracy_score(y, pred), 'balanced_accuracy': balanced_accuracy_score(y, pred)}

results = []
for sp in sorted([p.name for p in SPLITS_DIR.iterdir() if p.is_dir()]):
    res = evaluate(sp)
    results.append(res)
df_res = pd.DataFrame(results)
display(df_res)
df_res.plot.bar(x='species', y='balanced_accuracy', legend=False)
